# 04b \u2014 Pilot Campaign (small, for the demo)

**Purpose.** Run the **small pilot** guided/unguided campaigns (5 rounds x 20) for a fast end-to-end check and for the demo. The scaled reproduction (guided 25 rounds + unguided 24 rounds) lives in `04_campaign_full.ipynb`.

**Inputs / outputs**
- `real_benign_corpus/all/`
- panel images (picklescan/fickling/modelscan) + dynahug + oracle model dir

**Outputs**
- `data/regenbench_shadowpickle.db`
- `data/regenbench_campaign.db` (candidates, fitness, coverage)
- `docs/fuzzing-report-<run>.md`

In [ ]:
import sys, os
sys.path.insert(0, os.path.join(os.getcwd(), 'notebooks'))
from common import run, run_silent, sqlite, show, summary_line
run(["python3", "scripts/run_shadowpickle_baseline.py",
     "--candidates-per-family", "20", "--backend", "docker"])

In [ ]:
import sys, os
sys.path.insert(0, os.path.join(os.getcwd(), 'notebooks'))
from common import run, run_silent, sqlite, show, summary_line
run(["python3", "scripts/run_fuzzing_campaign.py", "--mode", "guided",
     "--rounds", "5", "--candidates-per-round", "20", "--replicate", "1",
     "--db", "data/regenbench_campaign.db", "--seed-corpus-dir", "real_benign_corpus/all", "--seed-cluster", "text-generation",
     "--attack-families", "gadget,overwritten,pypi_injected,external,indirect_chain",
     "--evasion-mode", "adaptive", "--fitness-mode", "oracle_aware",
     "--backend", "docker", "--seed", "42"])

In [ ]:
import sys, os
sys.path.insert(0, os.path.join(os.getcwd(), 'notebooks'))
from common import run, run_silent, sqlite, show, summary_line
run(["python3", "scripts/run_fuzzing_campaign.py", "--mode", "unguided",
     "--rounds", "5", "--candidates-per-round", "20", "--replicate", "1",
     "--db", "data/regenbench_campaign.db", "--seed-corpus-dir", "real_benign_corpus/all", "--seed-cluster", "text-generation",
     "--attack-families", "gadget,overwritten,pypi_injected,external,indirect_chain",
     "--evasion-mode", "random", "--fitness-mode", "current",
     "--backend", "docker", "--seed", "42"])

In [ ]:
import sys, os
sys.path.insert(0, os.path.join(os.getcwd(), 'notebooks'))
from common import run, run_silent, sqlite, show, summary_line
print("generated:", sqlite("SELECT COUNT(*) FROM candidates;"))
print("valid:    ", sqlite("SELECT COUNT(*) FROM campaign_fitness WHERE is_valid=1;"))
print("bypasses: ", sqlite("SELECT COUNT(*) FROM candidates c JOIN campaign_fitness f ON f.candidate_id=c.candidate_id WHERE f.is_valid=1 AND c.panel_verdict='all_benign';"))